<a href="https://colab.research.google.com/github/Qureshiii/PyTorch-Learning-Journey/blob/main/05_Data_Loading_and_Batching_with_DataLoaders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### DataSet and DataLoader

In [ ]:
from sklearn.datasets import make_classification
import torch

In [ ]:
X, y = make_classification(
    n_samples = 10,
    n_features= 2,
    n_informative= 2,
    n_redundant= 0,
    n_classes= 2,
    random_state= 42
)

In [ ]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [ ]:
X.shape

(10, 2)

In [ ]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [ ]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = features
    self.labels = labels

  def __len__(self):

    return self.features.shape[0]

  def __getitem__(self, index):

    return self.features[index], self.labels[index]

In [ ]:
dataset = CustomDataset(X, y)

In [ ]:
dataloader = DataLoader(dataset, batch_size= 2, shuffle=True)

In [ ]:
for batch_features, batch_labels in dataloader:

  print(batch_features)
  print(batch_labels)
  print('-'*50)

tensor([[-2.8954,  1.9769],
        [ 1.7273, -1.1858]])
tensor([0., 1.])
--------------------------------------------------
tensor([[ 1.7774,  1.5116],
        [-0.9382, -0.5430]])
tensor([1., 1.])
--------------------------------------------------
tensor([[ 1.8997,  0.8344],
        [ 1.0683, -0.9701]])
tensor([1., 1.])
--------------------------------------------------
tensor([[-1.1402, -0.8388],
        [-0.7206, -0.9606]])
tensor([0., 0.])
--------------------------------------------------
tensor([[-0.5872, -1.9717],
        [-1.9629, -0.9923]])
tensor([0., 0.])
--------------------------------------------------


### Improving previous Code

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-detection/refs/heads/master/data.csv')

In [ ]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [ ]:
scaler = StandardScaler()

# 1. Train data par fit aur transform dono karein
X_train = scaler.fit_transform(X_train)

# 2. Test data par SIRF transform karein (taaki train wala mean/std hi use ho)
X_test = scaler.transform(X_test)

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:

X_train_tensors = torch.from_numpy(X_train.astype(np.float32))
X_test_tensors = torch.from_numpy(X_test.astype(np.float32))
y_train_tensors = torch.from_numpy(y_train.astype(np.float32)).view(-1, 1)
y_test_tensors = torch.from_numpy(y_test.astype(np.float32)).view(-1, 1)

In [ ]:
train_dataset = CustomDataset(X_train_tensors, y_train_tensors)
test_dataset = CustomDataset(X_test_tensors, y_test_tensors)

In [ ]:
train_dataset[10]

(tensor([-0.0495, -0.5994, -0.1080, -0.1482, -2.0116, -0.9478, -0.8207, -0.8968,
         -0.0231, -1.0711, -0.8703, -1.1502, -0.7033, -0.5533, -1.2329, -0.6640,
         -0.6965, -1.0899, -0.8976, -0.6278, -0.2128, -0.6134, -0.1606, -0.2715,
         -1.7040, -0.3215, -0.6244, -0.7702, -0.3641, -0.3897]),
 tensor([0.]))

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [ ]:
import torch.nn as nn
class MySimpleNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()


  def forward(self,features):

    out = self.linear(features)
    out = self.sigmoid(out)
    return out

### Important Parameters

In [ ]:

learning_rate = 0.001
epochs = 100

In [ ]:
# create model
model = MySimpleNN(X_train_tensors.shape[1])

### Define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# define loss function
loss_function = nn.BCELoss()

### Training Pipeline

In [ ]:
for epoch in range(epochs):
   model.train()
   running_loss = 0.0

   for batch_features, batch_labels in train_loader:


      # 1. Forward pass
      y_pred = model(batch_features)

      # 2. Loss calculation
      loss = loss_function(y_pred, batch_labels.view(-1,1))

      # 3. clear grad
      optimizer.zero_grad()

      # 4. Backward pass
      loss.backward()

      # 5. parameter update across ALL layers
      optimizer.step()

      print(f"Epoch : {epoch + 1}, Loss : {loss.item():.4f}")

Epoch : 1, Loss : 0.5776
Epoch : 1, Loss : 0.6509
Epoch : 1, Loss : 0.5375
Epoch : 1, Loss : 0.5517
Epoch : 1, Loss : 0.5774
Epoch : 1, Loss : 0.5861
Epoch : 1, Loss : 0.6348
Epoch : 1, Loss : 0.6020
Epoch : 1, Loss : 0.6016
Epoch : 1, Loss : 0.5083
Epoch : 1, Loss : 0.6594
Epoch : 1, Loss : 0.5874
Epoch : 1, Loss : 0.5941
Epoch : 1, Loss : 0.6205
Epoch : 1, Loss : 0.4970
Epoch : 2, Loss : 0.5487
Epoch : 2, Loss : 0.5139
Epoch : 2, Loss : 0.6093
Epoch : 2, Loss : 0.5306
Epoch : 2, Loss : 0.5729
Epoch : 2, Loss : 0.6073
Epoch : 2, Loss : 0.5467
Epoch : 2, Loss : 0.5921
Epoch : 2, Loss : 0.6535
Epoch : 2, Loss : 0.4941
Epoch : 2, Loss : 0.5833
Epoch : 2, Loss : 0.6101
Epoch : 2, Loss : 0.5518
Epoch : 2, Loss : 0.6134
Epoch : 2, Loss : 0.4856
Epoch : 3, Loss : 0.5477
Epoch : 3, Loss : 0.5103
Epoch : 3, Loss : 0.5495
Epoch : 3, Loss : 0.6030
Epoch : 3, Loss : 0.5380
Epoch : 3, Loss : 0.5439
Epoch : 3, Loss : 0.5184
Epoch : 3, Loss : 0.6085
Epoch : 3, Loss : 0.4956
Epoch : 3, Loss : 0.6272


In [ ]:

# 6. Evaluation Loop
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        y_pred = model(batch_features)

        # Threshold set back to standard 0.5
        y_pred_class = (y_pred >= 0.5).float()

        correct += (y_pred_class == batch_labels).sum().item()
        total += batch_labels.size(0)

overall_accuracy = correct / total
print(f"\nFinal Test Accuracy: {overall_accuracy * 100:.2f}%")


Final Test Accuracy: 92.98%
